# Google MLCC: standalone Colab edition

Run the setup cell in a fresh Colab runtime, then run every cell from top to bottom. Setup downloads the original course data and installs compatible dependencies, including `google-ml-edu` (imported as `ml_edu`) for taxi/rice. No course ZIP is required. TensorFlow 2.21 and Keras 3.12+ adapt those exercises to current Python 3.13 Colab; the original TF 2.18/NumPy 2.0 pins do not support Python 3.13.

`USE_GOOGLE_DRIVE=True` saves measured results, models, and task execution evidence to `MyDrive/HaiLong_Colab_Results/01_google_mlcc`. Save this notebook in Drive and save it after running to retain cell outputs. The completion report requires every computational stage and full original data/epochs. Website quizzes and interactive exercises have separate progress.

Original source: https://raw.githubusercontent.com/google/eng-edu/main/ml/cc/exercises/numerical_data_stats.ipynb


In [ ]:
# Standalone Google MLCC setup. Start a fresh Colab runtime, then Run all.
# No ZIP or eng-edu clone is required. Full original data/tasks only.
from pathlib import Path
import os, sys, subprocess, importlib.metadata as metadata
import hashlib, json, shutil, urllib.request, platform, uuid
from datetime import datetime, timezone
from packaging.requirements import Requirement

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  # retained for compatibility; full completion requires False
if QUICK_RUN:
    raise ValueError('Use QUICK_RUN=False to execute the full original exercise.')
NOTEBOOK_NAME = '03_numerical_data_statistics.ipynb'
DEPENDENCY_PROFILE = 'basic'
REQUIREMENTS = ['numpy>=1.26,<3', 'pandas>=2.2,<3', 'matplotlib>=3.9,<4']
REQUIRED_TASKS = ['summary_statistics', 'outlier_interpretation']
DATA_SOURCES = [{'filename': 'california_housing_train.csv', 'url': 'https://download.mlcc.google.com/mledu-datasets/california_housing_train.csv', 'sha256': 'bb9ca4041a66e0ed691cb825188f16fdab518435c4950a080951be2fdfe30d4d', 'rows': 17000}]
ORIGINAL_SOURCE = 'https://raw.githubusercontent.com/google/eng-edu/main/ml/cc/exercises/numerical_data_stats.ipynb'
VISUALIZATION_ADAPTATION = None

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# TensorFlow 2.21 has Python 3.13 wheels; original TF 2.18 pins do not.
# Set Keras mode before importing TensorFlow or Keras.
if DEPENDENCY_PROFILE in ('google', 'fairness'):
    if DEPENDENCY_PROFILE == 'fairness':
        if any(n in sys.modules for n in ('tensorflow', 'keras', 'tf_keras')):
            if os.environ.get('TF_USE_LEGACY_KERAS') != '1':
                raise RuntimeError('Select Runtime > Disconnect and delete runtime, reconnect, then Run all: MinDiff needs legacy Keras selected before TensorFlow import.')
        os.environ['TF_USE_LEGACY_KERAS'] = '1'
    else:
        if os.environ.get('TF_USE_LEGACY_KERAS') == '1' and any(n in sys.modules for n in ('tensorflow', 'keras', 'tf_keras')):
            raise RuntimeError('Start a fresh runtime after fairness before running this Keras 3 notebook.')
        os.environ.pop('TF_USE_LEGACY_KERAS', None)
    os.environ['KERAS_BACKEND'] = 'tensorflow'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['COURSE_SMOKE'] = '0'
os.environ['WANDB_MODE'] = 'offline'

# google-ml-edu is the distribution that provides the ml_edu import.
IMPORT_NAMES = {'google-ml-edu':'ml_edu', 'tf-keras':'tf_keras',
    'tensorflow-model-remediation':'tensorflow_model_remediation',
    'scikit-learn':'sklearn'}
def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None
missing, loaded_changes = [], []
for text in REQUIREMENTS:
    req = Requirement(text)
    version = installed_version(req.name)
    if version is None or version not in req.specifier:
        missing.append(text)
        module = IMPORT_NAMES.get(req.name, req.name.replace('-', '_'))
        if module in sys.modules:
            loaded_changes.append(module)
if missing and IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *missing])
    if loaded_changes:
        raise RuntimeError('Installed required versions. Select Runtime > Restart session, then Run all again; old loaded modules: ' + ', '.join(loaded_changes))
elif missing:
    raise RuntimeError('Prepared for Colab. Missing compatible local packages: ' + ', '.join(missing))

if IN_COLAB:
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
else:
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Run in Colab or inside HaiLong_Colab_Exercises.')
COURSE_DIR = COURSE_ROOT / '01_google_mlcc'
COURSE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(COURSE_DIR)
DATA = COURSE_DIR / 'data'
DATA.mkdir(exist_ok=True)
DRIVE_DIR = None
if IN_COLAB and USE_GOOGLE_DRIVE:
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive', timeout_ms=300000)
    DRIVE_DIR = Path('/content/drive/MyDrive/HaiLong_Colab_Results/01_google_mlcc')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
for folder in ('results', 'models'):
    target = COURSE_DIR / folder
    if DRIVE_DIR is not None:
        (DRIVE_DIR / folder).mkdir(exist_ok=True)
        if not target.exists():
            target.symlink_to(DRIVE_DIR / folder, target_is_directory=True)
    target.mkdir(exist_ok=True)
RESULTS, MODELS = COURSE_DIR / 'results', COURSE_DIR / 'models'
for spec in DATA_SOURCES:
    target = DATA / spec['filename']
    if not target.is_file():
        temporary = target.with_suffix(target.suffix + '.download')
        print('Downloading original Google course data:', spec['url'])
        urllib.request.urlretrieve(spec['url'], temporary)
        if hashlib.sha256(temporary.read_bytes()).hexdigest() != spec['sha256']:
            raise ValueError('Downloaded dataset differs from verified original: ' + spec['filename'])
        temporary.replace(target)
    if hashlib.sha256(target.read_bytes()).hexdigest() != spec['sha256']:
        raise ValueError('Existing data differs from the original course dataset: ' + str(target))

EXECUTED_TASKS = {}
COLAB_CONTEXT = {'notebook': NOTEBOOK_NAME, 'run_id': str(uuid.uuid4()),
    'started_at_utc': datetime.now(timezone.utc).isoformat(),
    'in_hosted_colab': IN_COLAB, 'quick_run': False,
    'python': platform.python_version(),
    'versions': {Requirement(r).name: installed_version(Requirement(r).name) for r in REQUIREMENTS},
    'original_source': ORIGINAL_SOURCE, 'data_sources': DATA_SOURCES,
    'visualization_adaptation': VISUALIZATION_ADAPTATION}
def mark_task(name, **evidence):
    if name not in REQUIRED_TASKS:
        raise KeyError(name)
    EXECUTED_TASKS[name] = {'recorded_at_utc': datetime.now(timezone.utc).isoformat(), **evidence}
    checkpoint = {**COLAB_CONTEXT, 'last_completed_stage': name,
        'updated_at_utc': datetime.now(timezone.utc).isoformat(),
        'missing_tasks': sorted(set(REQUIRED_TASKS) - set(EXECUTED_TASKS)),
        'task_evidence': EXECUTED_TASKS, 'course_website_completion_confirmed': False}
    (RESULTS / (Path(NOTEBOOK_NAME).stem + '_progress.json')).write_text(
        json.dumps(checkpoint, indent=2, allow_nan=False), encoding='utf-8')
    if DRIVE_DIR is not None:
        for folder in ('results', 'models'):
            source, target = COURSE_DIR / folder, DRIVE_DIR / folder
            if source.resolve() != target.resolve():
                shutil.copytree(source, target, dirs_exist_ok=True)
def finish_run(context):
    missing_tasks = sorted(set(REQUIRED_TASKS) - set(EXECUTED_TASKS))
    if missing_tasks:
        raise RuntimeError('Required exercise stages did not run: ' + ', '.join(missing_tasks))
    report = {**context, 'finished_at_utc': datetime.now(timezone.utc).isoformat(),
        'full_original_computational_tasks_executed': True,
        'course_website_completion_confirmed': False,
        'required_tasks': REQUIRED_TASKS, 'task_evidence': EXECUTED_TASKS,
        'note': 'Execution evidence covers this notebook. It does not certify website quizzes, interactive exercises, or personal reflection.'}
    path = RESULTS / (Path(NOTEBOOK_NAME).stem + '_completion.json')
    path.write_text(json.dumps(report, indent=2, allow_nan=False), encoding='utf-8')
    if DRIVE_DIR is not None:
        for folder in ('results', 'models'):
            source, target = COURSE_DIR / folder, DRIVE_DIR / folder
            if source.resolve() != target.resolve():
                shutil.copytree(source, target, dirs_exist_ok=True)
        print('Saved computational progress and artifacts to:', DRIVE_DIR)
    print('Full original computational stages executed:', NOTEBOOK_NAME)
    print('Completion evidence:', path)
    return report
print('Setup ready:', NOTEBOOK_NAME, '| full course data |', DEPENDENCY_PROFILE)

Source: https://github.com/google/eng-edu/blob/main/ml/cc/exercises/numerical_data_stats.ipynb

Completed Apache-2.0 Google exercise with computed outlier evidence.


In [ ]:
#@title Copyright 2023 Google LLC. Double-click for license information.
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Colabs

Machine Learning Crash Course uses Colaboratories (Colabs) for all programming exercises. Colab is Google's implementation of [Jupyter Notebook](https://jupyter.org/). For more information about Colabs and how to use them, go to [Welcome to Colaboratory](https://research.google.com/colaboratory).

# Numerical data: Statistics on a dataset

This Colab programming exercise (first of two) is part of the Machine Learning Crash Course module [Working with numerical data](https://developers.google.com/machine-learning/crash-course/numerical-data).

## What to expect

In the section, [First steps with numerical data](https://developers.google.com/machine-learning/crash-course/numerical-data/first-steps), you learned how to do the following:
- Visualize your data in plots or graphs.
- Evaluate potential features and labels mathematically.
- Find [**outliers**](https://developers.google.com/machine-learning/glossary/#outliers) in the dataset.

This exercise takes you through the process of finding columns that contain blatant outliers, which you can then decide to keep in or delete from the dataset.

In [ ]:
from pathlib import Path
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('TF_NUM_INTRAOP_THREADS', '1')
os.environ.setdefault('TF_NUM_INTEROP_THREADS', '1')
# Colab setup above selects this course folder.
DATA = Path('data')
RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)

In [ ]:
# @title Setup - Import relevant modules

# The following code imports relevant modules that
# allow you to run the colab.
# If you encounter technical issues running some of the code sections
# that follow, try running this section again.

import pandas as pd

# The following lines adjust the granularity of reporting.
pd.options.display.max_rows = 10
pd.options.display.float_format = "{:.1f}".format

In [ ]:
training_df = pd.read_csv(DATA / 'california_housing_train.csv')

assert len(training_df) == 17000, 'Use the complete original California housing dataset.'

## Get basic statistics

In the following code section, the DataFrame `describe` method returns basic statistics on all the columns in the dataset, such as:

* `count` is the number of populated elements in this column. Ideally, every column contains the same value for `count`, but that's not always the case.
* `mean` is the traditional average of values in that column. We recommend comparing the `mean` to the median for each column. The **median** is the 50% row of the table.
* `std` is the standard deviation of the values in this column.
* `min`, `25%`, `50%`, `75%`, and `max` indicate values in the 0, 25, 50, 75, and 100th percentiles.

In [ ]:
display(training_df.describe())
training_df.describe().to_csv(RESULTS / 'housing_summary_statistics.csv')
mark_task('summary_statistics', rows=len(training_df))

### Task: Identify possible outliers

Based on the preceding statisics, do you see any columns that might contain outliers?

In [ ]:
# @title Solution (run this code block to view) { display-mode: "form" }

print("""The following columns might contain outliers:

  * total_rooms
  * total_bedrooms
  * population
  * households
  * possibly, median_income

In all of those columns:

  * the standard deviation is almost as high as the mean
  * the delta between 75% and max is much higher than the
      delta between min and 25%.""")

## Computed evidence for the outlier answer


In [ ]:
stats=training_df.describe().T
stats['upper_tail_gap']=stats['max']-stats['75%']
stats['lower_tail_gap']=stats['25%']-stats['min']
iqr=stats['75%']-stats['25%']
stats['outside_IQR_fences']=[int(((training_df[k]<stats.loc[k,'25%']-1.5*iqr[k])|
    (training_df[k]>stats.loc[k,'75%']+1.5*iqr[k])).sum()) for k in stats.index]
print(stats[['mean','std','upper_tail_gap','lower_tail_gap','outside_IQR_fences']].to_string())
stats.to_csv(RESULTS/'housing_outlier_audit.csv')
print('An extreme value is a candidate for investigation, not automatic proof of an error.')

# Quantify the source answer using the original data, rather than text alone.
suspected = ['total_rooms','total_bedrooms','population','households','median_income']
display(stats.loc[suspected, ['mean','std','upper_tail_gap','lower_tail_gap','outside_IQR_fences']])
print('These columns have long upper tails. Large values warrant investigation; the summary does not prove data-entry errors.')
mark_task('outlier_interpretation', candidate_columns=suspected, iqr_counts={k:int(stats.loc[k,'outside_IQR_fences']) for k in suspected})

## Save computational progress
Run the final cell after every exercise cell. It checks the required computational stages ran in this session, saves timestamped evidence, and synchronizes results/models to Drive. Save the notebook itself to retain its cell outputs. Website quiz/exercise progress is tracked separately by Google.


In [ ]:
finish_run(COLAB_CONTEXT)